# Application Tracing

Esta é uma documentação detalhada e didática da classe `ApplicationTracing`, projetada para facilitar a implementação de observabilidade em sistemas Python.

---

## Visão Geral

A classe `ApplicationTracing` atua como uma **Fachada (Facade)** para o gerenciamento de logs. Em vez de lidar diretamente com bibliotecas complexas ou configurações de baixo nível, o desenvolvedor utiliza esta classe para padronizar como as mensagens são registradas, exibidas e armazenadas.

O principal objetivo é garantir que todos os logs da aplicação sigam a mesma estrutura, facilitando a busca por falhas (troubleshooting) e a análise de dados em ferramentas como o MongoDB ou visualizadores de arquivos de log.

---

## Fluxo de Execução

O fluxo de funcionamento da classe segue estas etapas:

1. **Instanciação**: O desenvolvedor cria o objeto definindo as configurações globais (ex: se deve salvar no MongoDB, se deve mostrar metadados no console).
2. **Encapsulamento**: O construtor inicializa internamente o `LoggerEngine`, que é o motor real de processamento.
3. **Chamada de Método**: Ao chamar um método como `.INFO()` ou `.ERROR()`, os parâmetros passados (mensagem, metadados) são enviados ao motor.
4. **Sobrescrita Dinâmica**: Cada método de log permite sobrescrever as configurações globais momentaneamente (ex: o sistema não salva logs INFO por padrão, mas você pode forçar o salvamento de um log específico).
5. **Saída Multi-canal**: O log é processado e enviado simultaneamente para o console, arquivo local ou banco de dados, conforme configurado.

---

## Tabela de Métodos

| **Método** | **Nível de Severidade** | **Uso Recomendado** |
| --- | --- | --- |
| `__init__` | N/A | Configuração inicial e setup dos drivers de saída. |
| `INFO` | Informativo | Marcos de sucesso e fluxo normal (ex: "Processamento concluído"). |
| `DEBUG` | Baixo | Detalhes técnicos para desenvolvedores (ex: valores de variáveis). |
| `WARNING` | Médio | Situações inesperadas que não impedem a execução (ex: "API lenta"). |
| `ERROR` | Alto | Falhas em funcionalidades específicas (ex: "Falha ao enviar e-mail"). |
| `CRITICAL` | Urgente | Problemas que podem derrubar o sistema (ex: "Banco de dados offline"). |

---

## Arquitetura e Insights

- **Separação de Preocupações**: A classe não sabe *como* escrever no MongoDB; ela apenas delega essa responsabilidade para o `LoggerEngine`. Isso facilita a manutenção futura.
- **Flexibilidade Granular**: O design permite que você tenha um comportamento global, mas flexibilidade total em cada linha de código através dos argumentos opcionais nos métodos.
- **Observabilidade Estruturada**: O uso do argumento `metadata` (dicionário) incentiva o log estruturado (JSON), que é muito mais fácil de indexar e pesquisar do que simples strings de texto.
- **Rastreabilidade**: O parâmetro `log_id` permite correlacionar logs de uma mesma transação que atravessa diferentes funções ou arquivos.

---

## Documentação Técnica

## Classe `ApplicationTracing`

### Descrição

Classe responsável por abstrair e padronizar o uso de logs dentro da aplicação. Ela encapsula a lógica complexa de logging e fornece uma interface simplificada para o desenvolvedor.

### Argumentos do Construtor

- `log_id` (str): ID único para rastreamento de fluxo.
- `flag` (str): Marcador para categorizar logs.
- `file_name` (str): Contexto do arquivo de origem.
- `log_file_name` (str): Nome do arquivo `.log` de destino.
- `show_info_logs` (bool): Habilita/Desabilita logs de nível INFO no console.
- `show_metadata` (bool): Define se detalhes técnicos aparecem no console.
- `save_logs` (bool): Se `True`, grava em arquivo físico.
- `save_mongo` (bool): Se `True`, persiste no banco de dados.
- `format_metadata` (bool): Aplica formatação visual aos metadados.

---

## Métodos

## 1. INFO / DEBUG / WARNING / ERROR / CRITICAL

*(Os métodos compartilham a mesma assinatura para manter a consistência)*

**Descrição**

Registram uma mensagem no nível de severidade correspondente, processando o armazenamento e a exibição conforme as regras definidas na instância.

**Argumentos**

- `func_name` (Optional[str]): Nome da função onde o evento ocorreu.
- `message` (Optional[str]): Texto explicativo do log.
- `metadata` (Optional[Dict]): Dicionário com dados extras (ex: `{"user_id": 123}`).
- `save_logs` (Optional[bool]): Sobrescreve a configuração global de salvamento em arquivo.
- `save_mongo` (Optional[bool]): Sobrescreve a configuração global de salvamento no MongoDB.
- `show_info_logs` (Optional[bool]): Força a exibição ou ocultação de logs INFO.
- `show_metadata` (Optional[bool]): Força a exibição ou ocultação de metadados.

**Retornos**

- `None`: O método realiza o registro (efeito colateral) e não retorna dados.

**Raises**

- `Exception`: Pode propagar erros de conexão com banco de dados ou permissão de escrita em disco originados no `LoggerEngine`.

**Exemplos**

```bash
from src.tracing.tracing_core import ApplicationTracing

# Inicialização
tracer = ApplicationTracing(save_logs=True, log_id="TR-9952")

# Uso simples
tracer.INFO(
    func_name="create_user",
    message="App Init"
)

tracer.DEBUG(
    func_name="create_user",
    message="User created",
    metadata={"user": "Enzo"},
    #save_logs=False
    #show_metadata=False
)

tracer.WARNING(
    func_name="create_user",
    message="User created",
    metadata={"user": "Enzo"},
)

tracer.ERROR(
    func_name="create_user",
    message="User created",
    metadata={"user": "Enzo"},
)

tracer.CRITICAL(
    func_name="create_user",
    message="User created",
    metadata={"user": "Enzo"},
)

# Debug forçando salvamento no Mongo apenas para esta linha
tracer.DEBUG(message="Checkpont técnico", save_mongo=True)
```